In [ ]:
# install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load env variables
from dotenv import load_dotenv

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
# Helper functions for extending chat context window
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Chat function
def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text


# Make a Hello World request
# message = client.messages.create(
#    model=model,
#    max_tokens=1000,
#    messages=[
#        {
#            "role": "user",
#            "content": "What is the Hello World program"
#        }
#    ]
# )



In [ ]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "What is Minecraft"
add_user_message(messages, "What is Minecraft")

# Pass the list of messages into the 'chat' to get answer
answer = chat(messages)
answer
# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

# Add in the user's follow-up question
add_user_message(messages, "Write another sentence")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)
answer

In [ ]:
# Make an initial list of messages
message = []

# Use a 'while True' loop to run the chatbot forever
while True:
    # Get user input
    user_input = input("> ")
    print(">", user_input)

# Add user input to the list of messages
add_user_message(messages, user_input)

# Call Claude with the 'chat' function
answer = chat(messages)

# Add generated text to the list of messages
add_assistant_message(messages, answer)

# Print the generated text
print("---")
print(answer)
print("---")

In [ ]:
# System prompts
# Customizes Claudes tone and style of response
system_prompt="""
You are a coding wizard now.
"""

In [ ]:
# Helper functions for extending chat context window
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Chat function
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    return message.content[0].text

In [ ]:
messages = []

system="""
    You are a patient code tutor.
    Do not directly answer a student's questions.
    Guide them to a solution step by step.
    """

add_user_message(messages, "The message here")
answer = chat(messages, system=system)

answer

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)
for event in stream:
    print(event)

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="")    

stream.get_final_message()

In [ ]:
messages = []

add_user_message(messages, "Generate a very short eventbridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

In [ ]:
import json

json.loads(text.strip())

In [ ]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n ```bash")

text = chat(messages, stop_squences=["```"])
text.strip()

In [ ]:
from IPython.display import Markdown

Markdown(text)

In [ ]:
# Example prompt
prompt = f"""


{task}
"""

# Sample dataset
[
{
    "task": "chat prompt 1"
},
{
    "task": "chat prompt 2"
},
{
    "task": "chat prompt 3"
}
]

In [ ]:
import json 


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""


    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
# Run to generate dataset
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    """Merges the promp and test case inputs, then return the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Resposnd only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [ ]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [ ]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) /2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [ ]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and call run_test_case with each case"""
    results = []

    for test_case in dataset:
        results = run_test_case(test_case)
        results.append(results)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [ ]:
# Open and run dataset eval
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))